# Experiment Summary Notebook

Compare methods using anonymity score (1 = safe, 0 = risky) and usefulness score (0–80 pts).

## 1. Load data
Use helper functions to build the unified metrics table with anonymity/usefulness columns.

In [ ]:
from plot_experiment_summary import build_entries, load_dataframe, repo_root
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style='whitegrid')

root = repo_root()
entries = build_entries(root)
df = load_dataframe(entries).sort_values(['group', 'param_value']).reset_index(drop=True)
df[['variant', 'group', 'param_label', 'usefulness_score', 'anonymity_score_lr', 'anonymity_score_stats']]

## 2. Scatter plots
Horizontal = anonymity score (right is safer), vertical = usefulness score (higher is better).

In [ ]:
filtered = df[df['group'] != 'Differential Privacy'].reset_index(drop=True)
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
scatter_kwargs = dict(data=filtered, hue='group', style='group', s=140)
sns.scatterplot(x='anonymity_score_lr', y='usefulness_score', ax=axes[0], **scatter_kwargs)
sns.scatterplot(x='anonymity_score_stats', y='usefulness_score', ax=axes[1], legend=False, **scatter_kwargs)
for _, row in filtered.iterrows():
    axes[0].text(row['anonymity_score_lr'] + 0.01, row['usefulness_score'] + 0.5, row['param_label'], fontsize=8)
    axes[1].text(row['anonymity_score_stats'] + 0.01, row['usefulness_score'] + 0.5, row['param_label'], fontsize=8)
axes[0].set(title='Anonymity (model diff) vs usefulness', xlabel='Anonymity score (1 = safe, 0 = risky)', ylabel='Usefulness score (0–80 pts)', xlim=(0,1), ylim=(0,80))
axes[1].set(title='Anonymity (stat diff) vs usefulness', xlabel='Anonymity score (1 = safe, 0 = risky)', ylabel='Usefulness score (0–80 pts)', xlim=(0,1), ylim=(0,80))
handles, labels = axes[0].get_legend_handles_labels()
axes[0].legend(handles, labels, title='Method', fontsize=10, title_fontsize=12)
fig.tight_layout()
plt.show()

## 3. Per-method trade-offs
Left column shows usefulness, right column shows anonymity (Differential Privacy excluded).

In [ ]:
filtered = df[df['group'] != 'Differential Privacy'].reset_index(drop=True)
groups = filtered['group'].unique()
fig, axes = plt.subplots(len(groups), 2, figsize=(12, 3 * len(groups)), squeeze=False)
for row_idx, group in enumerate(groups):
    sub = filtered[filtered['group'] == group].sort_values('param_value')
    sns.lineplot(data=sub, x='param_value', y='usefulness_score', marker='o', color='tab:blue', ax=axes[row_idx, 0])
    sns.lineplot(data=sub, x='param_value', y='anonymity_score_lr', marker='o', color='tab:green', ax=axes[row_idx, 1])
    axes[row_idx, 0].set(title=f"{group} – Usefulness", xlabel='Parameter value', ylabel='Usefulness score (0–80 pts)', ylim=(0,80))
    axes[row_idx, 1].set(title=f"{group} – Anonymity", xlabel='Parameter value', ylabel='Anonymity score (1 = safe, 0 = risky)', ylim=(0,1))
    axes[row_idx, 0].grid(True, alpha=0.3)
    axes[row_idx, 1].grid(True, alpha=0.3)
fig.suptitle('Per-method parameter trade-offs', fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

## 4. Export CSV (optional)

In [ ]:
summary_dir = root / 'experiments' / 'summary_figures'
summary_dir.mkdir(parents=True, exist_ok=True)
output_csv = summary_dir / 'metrics_summary_from_notebook.csv'
df.to_csv(output_csv, index=False)
print(f'Saved: {output_csv}')